# Before You Leave: YOLOv8 Custom Model Training (Google Colab)
### Project: AI-Powered Personal Belongings Detection System
**Team: [Scuba]**
- Panyasiri Kiattiphumvarakun (116710905066-2)
- Phuwanai Khuyjad (116710905064-7)
- Worawut Phuangmalai (116710905077-9)

This notebook trains YOLOv8 on the 9 custom personal belongings classes:
1. `Phone`
2. `iPad_Tablet`
3. `Notebook`
4. `Charger_Cable`
5. `Wired_Earphones`
6. `Wireless_Earbuds`
7. `Wallet`
8. `Keys`
9. `Student_ID`

## Step 1: Install Ultralytics and Check GPU

In [ ]:
!pip install -q ultralytics roboflow
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## Step 2: Download Dataset (from Roboflow Universe or Upload Zip)
You can use a public personal items dataset or export from your Roboflow workspace.

In [ ]:
# Example: Download a personal belongings dataset from Roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("personal-items").project("before-you-leave")
# dataset = project.version(1).download("yolov8")

# Alternatively, create a dataset.yaml configuration file:
yaml_text = """
path: ./dataset
train: images/train
val: images/val
test: images/test

names:
  0: Phone
  1: iPad_Tablet
  2: Notebook
  3: Charger_Cable
  4: Wired_Earphones
  5: Wireless_Earbuds
  6: Wallet
  7: Keys
  8: Student_ID
"""
with open("data.yaml", "w") as f:
    f.write(yaml_text.strip())
print("data.yaml ready!")

## Step 3: Train YOLOv8 Model
We use `yolov8s.pt` with Transfer Learning for optimal accuracy.

In [ ]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO('yolov8s.pt')

# Train on custom dataset
results = model.train(
    data='data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    patience=15,
    augment=True,
    project='before_you_leave',
    name='yolov8s_belongings'
)

## Step 4: Validate and Evaluate Performance

In [ ]:
# Validate
metrics = model.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p:.4f}")
print(f"Recall: {metrics.box.r:.4f}")

## Step 5: Download `best.pt` to use in your Streamlit App

In [ ]:
from google.colab import files
best_weights = 'before_you_leave/yolov8s_belongings/weights/best.pt'
files.download(best_weights)
print("Downloaded best.pt! Place this file in your project folder and push to GitHub.")